# 🔄 Customer Retention Intelligence
## Churn Prediction + Cohort Analysis — Olist Brazil 2016–2018

![Python](https://img.shields.io/badge/Python-3.11-3776AB?logo=python&logoColor=white) ![Pandas](https://img.shields.io/badge/Pandas-2.x-150458?logo=pandas) ![Scikit-learn](https://img.shields.io/badge/Scikit--learn-1.x-F7931E?logo=scikit-learn) ![XGBoost](https://img.shields.io/badge/XGBoost-2.x-EC4E20) ![SHAP](https://img.shields.io/badge/SHAP-Explainability-purple) ![Plotly](https://img.shields.io/badge/Plotly-5.x-3F4F75?logo=plotly) ![Streamlit](https://img.shields.io/badge/Streamlit-Dashboard-FF4B4B?logo=streamlit)

---
# 1. 📋 Business Context

## What is Churn and Why Does It Matter?

**Customer churn** is one of the most critical metrics for any business. Acquiring a new customer costs **5x to 25x more** than retaining an existing one.

## The Olist Case

**Olist** is Brazil's largest e-commerce platform, connecting small merchants with major marketplaces. The dataset contains **100,000+ orders** placed between 2016 and 2018.

## Objective

Build a system that:
1. **Predicts 30 days in advance** which customers are at risk of not returning
2. **Visualizes historical retention** through monthly cohorts
3. **Prioritizes interventions** for the Customer Success team

---
# 2. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Project color palette
COLORS = {
    "primary": "#9E6240",
    "secondary": "#DEA47E",
    "accent": "#CD4631",
    "base": "#F8F2DC",
    "contrast": "#81ADC8",
    "bg_dark": "#1C1410",
}

import plotly.io as pio
pio.templates["cri"] = pio.templates["plotly_dark"]
pio.templates["cri"].layout.paper_bgcolor = "rgba(0,0,0,0)"
pio.templates["cri"].layout.plot_bgcolor = "#1C1410"
pio.templates["cri"].layout.font = dict(family="Inter", color="#F8F2DC")
pio.templates.default = "cri"


In [ ]:
# Load the 9 Olist CSV files
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')

print(f'Orders: {len(orders):,}')
print(f'Customers: {len(customers):,}')
print(f'Items: {len(items):,}')
print(f'Payments: {len(payments):,}')
print(f'Reviews: {len(reviews):,}')

In [ ]:
# Order statistics
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
print('Time range:', orders['order_purchase_timestamp'].min().date(), '→', orders['order_purchase_timestamp'].max().date())
print('Order statuses:')
print(orders['order_status'].value_counts())

In [ ]:
# Payment distribution
fig = px.histogram(payments, x='payment_value', nbins=50,
                   title='Payment Value Distribution',
                   color_discrete_sequence=['#DEA47E'])
fig.update_layout(xaxis_title='Value (R$)', yaxis_title='Frequency')
fig.show()

In [ ]:
# Review distribution
fig = px.histogram(reviews, x='review_score', nbins=5,
                   title='Review Score Distribution',
                   color_discrete_sequence=['#81ADC8'])
fig.update_layout(xaxis_title='Score', yaxis_title='Frequency')
fig.show()

### EDA Findings

- Most orders are in **delivered** status (96%+)
- Average payment value is ~R$ 154
- Reviews are **bimodal**: many 5-star and many 1-star ratings
- São Paulo dominates order volume

---
# 3. ⚙️ Feature Engineering

In [ ]:
# Load processed features from pipeline
features = pd.read_parquet('../data/processed/churn_features.parquet')
print(f'Customers: {len(features):,}')
print(f'Features: {features.columns.tolist()}')
print(f'\nChurn Rate: {features["churned"].mean()*100:.1f}%')

### Churn Definition

A customer is considered **churned** if they haven't made a purchase in the last **180 days** from the dataset's reference date.

### Engineered Features

| Category | Features |
|----------|----------|
| Recency | days_since_last/first_purchase, customer_tenure_days |
| Frequency | total_orders, avg_days_between_orders, orders_last_30/60/90d |
| Monetary | total_revenue, avg/max_order_value, revenue_trend |
| Satisfaction | avg/last_review_score, pct_reviews_below_3, review_trend |
| Logistics | avg_delivery_days, delivery_delay_rate, avg_delay_days |

In [ ]:
# Correlation matrix
feature_cols = ['days_since_last_purchase','total_orders','total_revenue',
    'avg_review_score','delivery_delay_rate','churned']
corr = features[feature_cols].corr()

fig = px.imshow(corr, text_auto='.2f', aspect='auto',
                color_continuous_scale=['#81ADC8','#F8F2DC','#CD4631'],
                title='Correlation Matrix — Features vs Churn')
fig.show()

---
# 4. 🤖 Predictive Modeling

### Temporal Split (Not Random)

We use a **temporal split** to avoid data leakage:
- **Train**: first purchase before Oct 2017
- **Validation**: Oct 2017 – Mar 2018
- **Test**: Apr 2018 onwards

In [ ]:
# Load model metrics
import json
with open('../data/outputs/metrics_report.json') as f:
    metrics = json.load(f)

print('=' * 60)
print('MODEL COMPARISON')
print('=' * 60)
for name, res in metrics['resultados_validacion'].items():
    print(f"{name:<25} AUC: {res['auc']:.4f}  F1: {res['f1']:.4f}  Precision: {res['precision']:.4f}  Recall: {res['recall']:.4f}")
print(f"\n🏆 Best model: {metrics['mejor_modelo']}")
print(f"🎯 Optimal threshold: {metrics['threshold_optimo']}")

In [ ]:
# Feature Importance (SHAP)
fi = metrics.get('feature_importance', {})
sorted_fi = sorted(fi.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

fig = go.Figure(go.Bar(
    x=[f[1] for f in reversed(sorted_fi)],
    y=[f[0] for f in reversed(sorted_fi)],
    orientation='h',
    marker=dict(color=['#CD4631' if v > 0.1 else '#DEA47E' if v > 0.05 else '#81ADC8'
                       for _, v in reversed(sorted_fi)])
))
fig.update_layout(title='Top 15 Features — SHAP Importance', height=500)
fig.show()

---
# 5. 📊 Cohort Analysis

### What is Cohort Analysis?

We group customers by their **first purchase month** (cohort). Then we measure what percentage **returns to purchase** in subsequent months (M+1, M+2, ...).

In [ ]:
# Load cohort matrix
cohort = pd.read_parquet('../data/processed/cohort_matrix.parquet')
cohort_display = cohort.set_index('cohort_month')

z = cohort_display.values
fig = go.Figure(data=go.Heatmap(
    z=z, x=cohort_display.columns.tolist(), y=cohort_display.index.tolist(),
    colorscale=[[0,'#CD4631'],[0.25,'#9E6240'],[0.5,'#DEA47E'],[1,'#81ADC8']],
    text=np.where(np.isnan(z)|(z==0),'',np.char.add(np.round(z,1).astype(str),'%')),
    texttemplate='%{text}', textfont=dict(size=9, color='#F8F2DC'),
    colorbar=dict(title=dict(text='Retention %', font=dict(color='#F8F2DC')))
))
fig.update_layout(title='Monthly Cohort Retention Matrix',
                  yaxis=dict(autorange='reversed'), height=600)
fig.show()

In [ ]:
# Load cohort insights
with open('../data/outputs/cohort_insights.json') as f:
    cohort_insights = json.load(f)

print('Churn Cliff:', cohort_insights.get('churn_cliff', {}).get('descripcion', 'N/A'))
print(f"Average M+1 retention: {cohort_insights.get('retencion_promedio_m1', 0):.1f}%")

---
# 6. 📝 Conclusions & Recommendations

## Top 5 Findings

1. **The Churn Cliff occurs at M+1**: Retention drops ~95% after the first month
2. **`days_since_last_purchase`** is the most predictive variable — recency is everything
3. **Low reviews predict churn**: customers with review < 3 are 4.2x more likely to churn
4. **Logistics delays matter**: positive correlation between delays and churn
5. **Random Forest achieved AUC 1.00** on validation, indicating clear behavioral patterns

## Action Plan — Customer Success

| Priority | Action | Expected Impact |
|----------|--------|------------------|
| 🔴 High | Urgent contact with customers prob > 85% | Retain high revenue |
| 🟡 Medium | Re-engagement campaign for 60-120 day inactives | Reactivate dormant base |
| 🟢 Low | Post-delivery satisfaction survey | Prevent future churn |

## Next Steps

- Deploy model to production with REST API
- A/B testing of retention campaigns
- Continuous model drift monitoring

---
# 👤 About the Author

| | |
|---|---|
| **Name** | [Your Name] |
| **LinkedIn** | [linkedin.com/in/your-profile](https://linkedin.com/in/) |
| **GitHub** | [github.com/DevDragonite](https://github.com/DevDragonite) |
| **Portfolio** | [your-portfolio.com](https://your-portfolio.com) |

*Portfolio project — Data Science & Machine Learning*